In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
## Create Flag Parameter
dbutils.widgets.text('incremental_flag','o')

In [0]:
# get the value of the incremental_flag (will be used for incremental load and intial load)
# incremental flag is a string
# incremental_flad is used to determine if the data is incremental or initial load
incremental_flag = dbutils.widgets.get('incremental_flag')
print(incremental_flag)


# **Creating Dimension Model**

In [0]:
df_src = spark.sql(
    """
    SELECT DISTINCT(Model_ID) as Model_ID, Model_Category
    FROM PARQUET.`abfss://silver@carsalesdata.dfs.core.windows.net/carsales`
    """
)

df_src.display()

### dim_model Sink - Initial & Incremental

In [0]:
# this query is never true, so it will not return any data but it will only return the SCHEMA of the dimension table that we want to create
# this condition is only for initial laod
# if the table does not exist, I want to get the schema
# if the table exists (incremental load), bring all the data (which will be used in the left join)

if not spark.catalog.tableExists('cars_catalog.gold.dim_model'):
    df_sink = spark.sql(
        '''
            SELECT 1 AS dim_model_key,
            Model_ID, Model_Category FROM PARQUET.`abfss://silver@carsalesdata.dfs.core.windows.net/carsales`
            WHERE 1=0
        '''
    )
else:
    df_sink = spark.sql(
        '''
        SELECT dim_model_key,Model_ID, Model_Category
        FROM cars_catalog.gold.dim_model
        '''
    )

df_sink.display()


## Filtering new records & old records

In [0]:
# df_src is the left table, df_sink left table in the left join. 
df_filter = df_src.join(df_sink,df_src.Model_ID == df_sink.Model_ID,'left').select(df_src.Model_ID,df_src.Model_Category,df_sink.dim_model_key)

df_filter.display()

In [0]:
df_filter_old = df_filter.filter(df_filter.dim_model_key.isNotNull())
df_filter_old.display()

In [0]:
df_filter_new = df_filter.filter(df_filter.dim_model_key.isNull()).select('Model_ID','Model_Category')
df_filter_new.display()

### Create Surrogate Key
A surrogate key also called a **synthetic primary key**, is generated when a new record is inserted into a table automatically by a database that can be declared as the primary key of that table.
### Fetch the max Surrogate Key from existing 

In [0]:
#we will use the incremental flag
if (incremental_flag == '0'):
    max_value = 1
else:
    if spark.catalog.tableExists('cars_catalog.gold.dim_model'):
        max_value_df = spark.sql("SELECT MAX(dim_model_key) FROM cars_catalog.gold.dim_model")
        max_value = max_value_df.collect()[0][0]

if spark.catalog.tableExists('cars_catalog.gold.dim_model'):
    max_value_df = spark.sql("SELECT MAX(dim_model_key) FROM cars_catalog.gold.dim_model")
    max_value = max_value_df.collect()[0][0]
else:
    max_value = 1

In [0]:
from pyspark.sql import functions as F
df_filter_new = df_filter_new.withColumn('dim_model_key',max_value + F.monotonically_increasing_id())

df_filter_new.display()

In [0]:
df_final = df_filter_new.union(df_filter_old)
df_final.display()

## SCD Type - 1 (UPSERT)

In [0]:
from delta.tables import DeltaTable

#Incremental Run

if spark.catalog.tableExists('cars_catalog.gold.dim_model'):
    delta_table = DeltaTable.forPath(spark,"abfss://gold@carsalesdata.dfs.core.windows.net/dim_model")
    delta_table.alias("target").merge(df_final.alias("source"),"target.dim_model_key = source.dim_model_key")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()
else:
    df_final.write.format("delta")\
        .mode("overwrite")\
        .option("path","abfss://gold@carsalesdata.dfs.core.windows.net/dim_model")\
        .saveAsTable("cars_catalog.gold.dim_model")
    

In [0]:
%sql
Select * FROM cars_catalog.gold.dim_model